# Week 04 Practice: Lakehouse Design with Spark + Iceberg + MinIO

Four progressive parts:

1. **Storage formats** — CSV vs JSON vs Parquet on object storage (MinIO/S3)
2. **Apache Iceberg basics** — create tables on S3, snapshots, time travel
3. **Schema evolution** — add/rename columns without rewriting data files
4. **Medallion architecture** — bronze → silver → gold pipeline on Iceberg

Run all cells **in order**. Each section builds on the previous one.

**MinIO Console:** Open [http://localhost:9001](http://localhost:9001) (admin / password) to browse files at any time.

---

**Reference book:** *Designing Data-Intensive Applications*, 2nd Ed. — Ch. 4 (Storage) & Ch. 5 (Encoding)

---
## Setup

The Spark session is configured to:
- Connect to **MinIO** as S3-compatible object storage (`s3a://` protocol)
- Use an **Iceberg Hadoop catalog** that stores tables at `s3a://warehouse/`

This mirrors a production lakehouse where Spark reads/writes to S3 — the only difference is MinIO runs locally in Docker instead of in the cloud.

In [14]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.types import *
import os, time

S3_ENDPOINT = "http://localhost:9000"
S3_BUCKET   = "s3a://warehouse"

spark = (
    SparkSession.builder
    .appName("bdm_week04_lakehouse")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "4")
    # ---- S3 / MinIO configuration ----
    .config("spark.hadoop.fs.s3a.endpoint", S3_ENDPOINT)
    .config("spark.hadoop.fs.s3a.access.key", "admin")
    .config("spark.hadoop.fs.s3a.secret.key", "password")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")        # required for MinIO
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    # ---- Iceberg configuration ----
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    .config("spark.sql.catalog.lakehouse", "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.lakehouse.type", "hadoop")
    .config("spark.sql.catalog.lakehouse.warehouse", S3_BUCKET)
    .config("spark.sql.defaultCatalog", "lakehouse")
    .config("spark.hadoop.fs.s3a.aws.credentials.provider",
        "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f"Spark {spark.version}")
print(f"S3 endpoint: {S3_ENDPOINT}")
print(f"Warehouse:   {S3_BUCKET}")
print("Ready! Open http://localhost:9001 to browse MinIO.")

Spark 4.1.0
S3 endpoint: http://localhost:9000
Warehouse:   s3a://warehouse
Ready! Open http://localhost:9001 to browse MinIO.


---
## Part 1 — Storage Formats: CSV vs JSON vs Parquet

### Why does the storage format matter?

From Chapter 4 of DDIA:
- **Row-oriented** formats (CSV, JSON) store each record contiguously. Good for OLTP, bad for analytics.
- **Column-oriented** formats (Parquet, ORC) store each column contiguously. Great for analytics — read only the columns you need, compress similar values together.

### Why object storage?

In a lakehouse, data lives on **object storage** (S3, GCS, Azure Blob) — not on local disks attached to compute nodes. This gives you:
- **Separation of storage and compute** — scale each independently
- **Durability** — object stores replicate data across zones/regions
- **Cost** — object storage is 5–10× cheaper than SSD-attached compute
- **Shared access** — multiple engines (Spark, Trino, DuckDB) read the same files

MinIO simulates S3 locally. After writing files, open [http://localhost:9001](http://localhost:9001) → browse the `warehouse` bucket to see them.

Let's write the same dataset in three formats and compare.

In [15]:
# Generate a synthetic sales dataset — 100,000 rows.
import random
from datetime import datetime, timedelta

random.seed(42)
base_date = datetime(2024, 1, 1)
countries = ["Estonia", "Finland", "Latvia", "Lithuania", "Sweden", "Germany", "Poland", "Norway"]
products  = [f"product-{i:03d}" for i in range(1, 51)]  # 50 products

rows = []
for i in range(100_000):
    rows.append((
        i + 1,
        (base_date + timedelta(days=random.randint(0, 364))).strftime("%Y-%m-%d"),
        random.choice(countries),
        random.choice(products),
        random.randint(1, 20),
        round(random.uniform(5.0, 500.0), 2),
    ))

schema = StructType([
    StructField("order_id",    IntegerType()),
    StructField("order_date",  StringType()),
    StructField("country",     StringType()),
    StructField("product_id",  StringType()),
    StructField("quantity",    IntegerType()),
    StructField("unit_price",  DoubleType()),
])

sales_df = spark.createDataFrame(rows, schema)
print(f"Generated {sales_df.count():,} rows")
sales_df.show(5)

Generated 100,000 rows
+--------+----------+-------+-----------+--------+----------+
|order_id|order_date|country| product_id|quantity|unit_price|
+--------+----------+-------+-----------+--------+----------+
|       1|2024-11-23|Finland|product-002|       9|    126.22|
|       2|2024-03-12|Finland|product-044|      18|     48.03|
|       3|2024-08-04|Estonia|product-002|       3|    113.23|
|       4|2024-09-15|Estonia|product-036|       7|    359.43|
|       5|2024-12-25| Poland|product-015|      15|    296.69|
+--------+----------+-------+-----------+--------+----------+
only showing top 5 rows


In [16]:
# Write the same data in three formats — all to MinIO (S3).
# After this cell, open http://localhost:9001 → warehouse bucket → format_comparison/

BASE = f"{S3_BUCKET}/format_comparison"

sales_df.write.mode("overwrite").csv(f"{BASE}/csv", header=True)
sales_df.write.mode("overwrite").json(f"{BASE}/json")
sales_df.write.mode("overwrite").parquet(f"{BASE}/parquet")

print("Written to MinIO:")
print(f"  {BASE}/csv/")
print(f"  {BASE}/json/")
print(f"  {BASE}/parquet/")
print()
print("Go to http://localhost:9001 → warehouse → format_comparison to browse the files!")

NumberFormatException: For input string: "60s"

In [ ]:
# Compare file sizes on object storage.
# We use Hadoop's FileSystem API to list files and sum their sizes.

def s3_dir_size(path):
    """Sum file sizes in an S3 directory using Hadoop FileSystem API."""
    hadoop_conf = spark.sparkContext._jsc.hadoopConfiguration()
    fs = spark.sparkContext._jvm.org.apache.hadoop.fs.FileSystem.get(
        spark.sparkContext._jvm.java.net.URI(path), hadoop_conf
    )
    status = fs.listStatus(spark.sparkContext._jvm.org.apache.hadoop.fs.Path(path))
    total = sum(s.getLen() for s in status if not s.isDirectory())
    return total

def fmt_size(size_bytes):
    if size_bytes < 1024:
        return f"{size_bytes} B"
    elif size_bytes < 1024 * 1024:
        return f"{size_bytes / 1024:.1f} KB"
    else:
        return f"{size_bytes / (1024*1024):.1f} MB"

print(f"{'Format':<12} {'Size':>10}  {'vs Parquet':>12}")
print("-" * 38)

sizes = {}
for fmt in ["csv", "json", "parquet"]:
    sz = s3_dir_size(f"{BASE}/{fmt}")
    sizes[fmt] = sz
    ratio = f"{sz / sizes.get('parquet', sz):.1f}×" if 'parquet' in sizes else ""
    print(f"{fmt:<12} {fmt_size(sz):>10}  {ratio:>12}")

print()
print(f"Parquet is ~{sizes['csv'] / sizes['parquet']:.0f}× smaller than CSV")
print(f"Parquet is ~{sizes['json'] / sizes['parquet']:.0f}× smaller than JSON")
print("\nThis is column compression at work (dictionary, RLE, delta encoding).")

In [ ]:
# Query performance comparison — reading from MinIO (S3).
# Parquet benefits from column pruning: only reads the columns needed.

def timed_query(name, df):
    start = time.time()
    result = (
        df
        .withColumn("revenue", F.col("quantity") * F.col("unit_price"))
        .groupBy("country")
        .agg(F.round(F.sum("revenue"), 2).alias("total_revenue"))
        .orderBy(F.desc("total_revenue"))
    )
    result.collect()  # force execution
    elapsed = time.time() - start
    print(f"{name:<12} {elapsed:.3f}s")
    return result

print(f"{'Format':<12} {'Time':>8}")
print("-" * 22)

csv_s3 = spark.read.csv(f"{BASE}/csv", header=True, inferSchema=True)
timed_query("CSV", csv_s3)

json_s3 = spark.read.json(f"{BASE}/json")
timed_query("JSON", json_s3)

pq_s3 = spark.read.parquet(f"{BASE}/parquet")
result = timed_query("Parquet", pq_s3)

print("\nParquet is fastest — it reads only 3 columns (country, quantity, unit_price)")
print("instead of loading all 6 columns like CSV/JSON must.")
print("\nResult:")
result.show()

### 🔍 Checkpoint — Browse MinIO

Open [http://localhost:9001](http://localhost:9001) → login with `admin` / `password` → click the **warehouse** bucket.

Navigate to `format_comparison/` and compare:
- `csv/` — many part files, each stores full rows as text
- `json/` — even larger, each row is a full JSON object
- `parquet/` — much smaller files, binary columnar format

This is exactly what your S3 bucket would look like in AWS or GCP.

---
## Part 2 — Apache Iceberg Basics

### What is Iceberg?

Apache Iceberg is a **table format** — a metadata layer on top of Parquet/ORC files that adds:
- **ACID transactions** — no partial writes, no corrupted reads
- **Snapshot isolation** — readers see a consistent view even while writers are active
- **Time travel** — query any previous version of the table
- **Schema evolution** — safely change the schema without rewriting data
- **Partition evolution** — change partitioning without rewriting data

This is the **transaction layer** that turns a data lake into a **lakehouse** (DDIA Ch. 4).

### Where does data live?

All Iceberg tables are stored on MinIO at `s3a://warehouse/`. When you create a table, Iceberg writes:
- **Data files** (Parquet) — the actual column data
- **Metadata files** (JSON + Avro) — snapshots, manifests, schema history

You can browse all of this in the MinIO Console.

In [ ]:
# Create a database (namespace) in our Iceberg catalog.
spark.sql("CREATE DATABASE IF NOT EXISTS lakehouse.practice")
spark.sql("USE lakehouse.practice")
print("Using database: lakehouse.practice")

In [ ]:
# Create our first Iceberg table.
# The data will be stored as Parquet files on MinIO.

spark.sql("""
    CREATE OR REPLACE TABLE sales (
        order_id    INT,
        order_date  DATE,
        country     STRING,
        product_id  STRING,
        quantity    INT,
        unit_price  DOUBLE
    )
    USING iceberg
""")

print("Iceberg table 'sales' created on s3a://warehouse/practice.db/sales/")
print()
print("Check MinIO Console → warehouse → practice.db → sales → metadata/")
print("You should see the first metadata file (no data files yet).")

In [ ]:
# Insert data into the Iceberg table.

sales_typed = sales_df.withColumn("order_date", F.to_date("order_date"))
sales_typed.writeTo("sales").append()

count = spark.table("sales").count()
print(f"Inserted {count:,} rows.")
print()
print("Check MinIO → warehouse → practice.db → sales → data/")
print("You'll now see Parquet data files alongside the metadata.")

In [ ]:
# Query the Iceberg table — just like a regular Spark SQL table.
spark.sql("""
    SELECT country,
           COUNT(*) AS orders,
           ROUND(SUM(quantity * unit_price), 2) AS total_revenue
    FROM sales
    GROUP BY country
    ORDER BY total_revenue DESC
""").show()

In [ ]:
# Inspect Iceberg snapshots.
# Each write operation creates a new snapshot — this is how time travel works.

print("=== Snapshots ===")
spark.sql("SELECT snapshot_id, committed_at, operation FROM lakehouse.practice.sales.snapshots").show(truncate=False)

In [ ]:
# Insert a second batch — creates snapshot #2.
extra_rows = [
    (100001, "2024-12-25", "Estonia", "product-001", 100, 999.99),
    (100002, "2024-12-25", "Estonia", "product-002", 200, 499.99),
]
extra_df = spark.createDataFrame(extra_rows, schema).withColumn("order_date", F.to_date("order_date"))
extra_df.writeTo("sales").append()

print(f"Total rows now: {spark.table('sales').count():,}")
print()
print("=== Snapshots (now 2) ===")
spark.sql("SELECT snapshot_id, committed_at, operation FROM lakehouse.practice.sales.snapshots").show(truncate=False)

In [ ]:
# Time travel — query the table AS OF the first snapshot.

snapshots = spark.sql(
    "SELECT snapshot_id FROM lakehouse.practice.sales.snapshots ORDER BY committed_at"
).collect()
first_snapshot = snapshots[0]["snapshot_id"]

old_count = spark.sql(f"SELECT COUNT(*) AS cnt FROM sales VERSION AS OF {first_snapshot}").collect()[0]["cnt"]
new_count = spark.table("sales").count()

print(f"Rows at snapshot 1 (time travel): {old_count:,}")
print(f"Rows at current snapshot:         {new_count:,}")
print(f"Difference:                       {new_count - old_count}")
print()
print("Time travel reads the OLD Parquet files — no data was copied or restored.")
print("Iceberg just points the query at a different set of data files via the snapshot metadata.")

In [ ]:
# Inspect the Parquet data files tracked by Iceberg.
print("=== Data Files ===")
spark.sql("""
    SELECT file_path, file_format, record_count,
           file_size_in_bytes AS size_bytes
    FROM lakehouse.practice.sales.files
""").show(truncate=50)

print("These are the actual Parquet files on MinIO. You can browse them in the Console.")

---
## Part 3 — Schema Evolution

From DDIA Chapter 5: applications change over time, and so do their schemas. Iceberg handles this at the table format level:

- **Add columns** — old Parquet files return NULL for the new column
- **Rename columns** — Iceberg tracks columns by internal ID, not name
- **Widen types** — e.g., int → long
- **Drop columns** — metadata stops exposing them

All **without rewriting existing data files** on MinIO. This is the Iceberg equivalent of backward/forward compatibility from Ch. 5.

In [ ]:
# Add a new column.
# Old Parquet files on MinIO are NOT rewritten — they just don't have this column.
# Iceberg returns NULL for old rows.

spark.sql("ALTER TABLE sales ADD COLUMNS (revenue DOUBLE)")

print("Column 'revenue' added.")
print()
spark.sql("SELECT order_id, country, quantity, unit_price, revenue FROM sales LIMIT 5").show()
print("revenue = NULL for old rows — the Parquet files on MinIO were NOT rewritten.")
print("Verify: check the file timestamps in MinIO Console — they haven't changed.")

In [ ]:
# Insert new rows WITH the revenue column populated.
# This demonstrates backward compatibility: new data has the field, old data doesn't.

new_rows = spark.createDataFrame([
    (200001, "2025-01-15", "Estonia", "product-010", 5, 49.99, 249.95),
    (200002, "2025-01-15", "Finland", "product-020", 3, 99.99, 299.97),
], "order_id INT, order_date STRING, country STRING, product_id STRING, quantity INT, unit_price DOUBLE, revenue DOUBLE")
new_rows = new_rows.withColumn("order_date", F.to_date("order_date"))
new_rows.writeTo("sales").append()

print("New rows with revenue:")
spark.sql("SELECT * FROM sales WHERE revenue IS NOT NULL").show()

In [ ]:
# Rename a column — Iceberg tracks by internal ID, not name.
spark.sql("ALTER TABLE sales RENAME COLUMN unit_price TO price_per_unit")

print("Renamed: unit_price → price_per_unit")
print()
spark.sql("SELECT order_id, price_per_unit, revenue FROM sales WHERE revenue IS NOT NULL").show()

print("The old Parquet files on MinIO still contain 'unit_price' as the column name.")
print("Iceberg resolves it by column ID at read time — no file rewrite needed.")

In [ ]:
# View all snapshots — each schema change + data write is tracked.
print("=== All snapshots ===")
spark.sql("""
    SELECT snapshot_id, committed_at, operation
    FROM lakehouse.practice.sales.snapshots
    ORDER BY committed_at
""").show(truncate=False)

---
## Part 4 — Medallion Architecture (Bronze → Silver → Gold)

| Layer | Purpose | Data quality |
|-------|---------|-------------|
| **Bronze** | Raw ingestion — store everything as-is | Low (raw strings) |
| **Silver** | Cleaned, typed, validated | Medium (correct types, nulls handled) |
| **Gold** | Business aggregates, ready for BI/ML | High (pre-computed metrics) |

Each layer is an Iceberg table on MinIO with full ACID and time travel.

We'll simulate ingesting raw JSON sensor data and transforming it through the layers.

In [ ]:
# Generate raw sensor data — simulates what Kafka would deliver.
import json as json_lib

random.seed(99)
base = datetime(2025, 3, 20, 8, 0, 0)
raw_records = []

for i in range(500):
    sensor = f"sensor-{random.randint(1, 10):02d}"
    ts = base + timedelta(minutes=random.randint(0, 1440))
    temp = round(random.gauss(21.0, 3.0), 2)
    humidity = round(random.gauss(55.0, 10.0), 1)

    # Inject dirty data (5% chance: missing temperature)
    if random.random() < 0.05:
        temp = None

    payload = json_lib.dumps({
        "sensor_id": sensor,
        "timestamp": ts.strftime("%Y-%m-%dT%H:%M:%S"),
        "temperature": temp,
        "humidity": humidity,
    })
    raw_records.append((payload, datetime.now().strftime("%Y-%m-%dT%H:%M:%S")))

raw_df = spark.createDataFrame(raw_records, "raw_value STRING, ingest_time STRING")
print(f"Generated {raw_df.count()} raw sensor records (5% have missing temperature)")
raw_df.show(3, truncate=60)

In [ ]:
# === BRONZE LAYER ===
# Store raw JSON strings exactly as received. No parsing, no validation.
# In production: Kafka readStream → append to Bronze Iceberg table.

spark.sql("""
    CREATE OR REPLACE TABLE bronze_sensors (
        raw_value    STRING,
        ingest_time  TIMESTAMP
    ) USING iceberg
""")

raw_df.withColumn("ingest_time", F.to_timestamp("ingest_time")).writeTo("bronze_sensors").append()

print(f"Bronze: {spark.table('bronze_sensors').count()} rows on MinIO")
spark.sql("SELECT * FROM bronze_sensors LIMIT 3").show(truncate=60)

In [ ]:
# === SILVER LAYER ===
# Parse JSON, cast types, filter bad records.

spark.sql("""
    CREATE OR REPLACE TABLE silver_sensors (
        sensor_id    STRING,
        event_time   TIMESTAMP,
        temperature  DOUBLE,
        humidity     DOUBLE
    ) USING iceberg
""")

sensor_schema = "sensor_id STRING, timestamp STRING, temperature DOUBLE, humidity DOUBLE"

silver_df = (
    spark.table("bronze_sensors")
    .select(F.from_json("raw_value", sensor_schema).alias("d"))
    .select(
        "d.sensor_id",
        F.to_timestamp("d.timestamp").alias("event_time"),
        "d.temperature",
        "d.humidity",
    )
    .filter(F.col("temperature").isNotNull())
    .filter(F.col("sensor_id").isNotNull())
)

silver_df.writeTo("silver_sensors").append()

bronze_cnt = spark.table("bronze_sensors").count()
silver_cnt = spark.table("silver_sensors").count()
print(f"Bronze: {bronze_cnt} rows")
print(f"Silver: {silver_cnt} rows ({bronze_cnt - silver_cnt} bad records filtered)")
spark.sql("SELECT * FROM silver_sensors ORDER BY event_time LIMIT 5").show(truncate=False)

In [ ]:
# === GOLD LAYER ===
# Hourly averages per sensor — ready for dashboards.

spark.sql("""
    CREATE OR REPLACE TABLE gold_hourly_sensors (
        sensor_id      STRING,
        hour           TIMESTAMP,
        avg_temp       DOUBLE,
        avg_humidity   DOUBLE,
        reading_count  LONG
    ) USING iceberg
""")

gold_df = (
    spark.table("silver_sensors")
    .withColumn("hour", F.date_trunc("hour", "event_time"))
    .groupBy("sensor_id", "hour")
    .agg(
        F.round(F.avg("temperature"), 2).alias("avg_temp"),
        F.round(F.avg("humidity"), 1).alias("avg_humidity"),
        F.count("*").alias("reading_count"),
    )
)

gold_df.writeTo("gold_hourly_sensors").append()

print(f"Gold: {spark.table('gold_hourly_sensors').count()} hourly aggregates")
spark.sql("""
    SELECT * FROM gold_hourly_sensors
    ORDER BY hour, sensor_id
    LIMIT 10
""").show(truncate=False)

In [ ]:
# Summary: the medallion pipeline.
print("=== Medallion Architecture Summary ===")
print(f"{'Layer':<10} {'Table':<25} {'Rows':>8}")
print("-" * 45)
for layer, table in [("Bronze", "bronze_sensors"), ("Silver", "silver_sensors"), ("Gold", "gold_hourly_sensors")]:
    cnt = spark.table(table).count()
    print(f"{layer:<10} {table:<25} {cnt:>8,}")

print()
print("All tables are Iceberg on MinIO (S3). Browse them:")
print("  http://localhost:9001 → warehouse → practice.db/")
print()
print("Each table has ACID transactions, snapshots, and time travel.")

In [ ]:
# List all tables in our lakehouse.
spark.sql("SHOW TABLES IN lakehouse.practice").show(truncate=False)

---
## Exercises

You are free to use any resource (documentation, AI tools, etc.).

---
### Exercise 1 — Partitioned Iceberg table + snapshot rollback

1. Create a new Iceberg table `sales_partitioned` **partitioned by `country`**.
   - Hint: `PARTITIONED BY (country)` in CREATE TABLE.
2. Insert the `sales_typed` DataFrame.
3. Show total revenue by country.
4. `DELETE FROM sales_partitioned WHERE country = 'Estonia'`
5. Verify Estonia rows are gone.
6. Use `CALL lakehouse.system.rollback_to_snapshot()` to **undo the delete**.
   - Find the pre-delete snapshot ID from `sales_partitioned.snapshots`.
   - `CALL lakehouse.system.rollback_to_snapshot('practice.sales_partitioned', <id>)`
7. Verify Estonia rows are back.

**Bonus:** Browse MinIO before and after the delete — how does the partitioned layout differ from unpartitioned?

In [ ]:
# Exercise 1 — your answer


---
### Exercise 2 — Compression comparison on S3

1. Write `sales_df` as Parquet with **Snappy** (default) and **GZIP** compression to MinIO.
   - `s3a://warehouse/compression/snappy/` and `s3a://warehouse/compression/gzip/`
   - Hint: `.option("compression", "gzip")`
2. Compare file sizes using the `s3_dir_size()` helper from Part 1.
3. Run the revenue-by-country query on both and compare times.
4. Answer in a markdown cell:
   - Which compression gives better space savings?
   - Which is faster to query? Why might there be a trade-off?
   - When would you choose GZIP over Snappy in a lakehouse?

In [ ]:
# Exercise 2 — your answer


---
## Further reading

- [Apache Iceberg — Spark Quickstart](https://iceberg.apache.org/spark-quickstart/)
- [Iceberg Spark DDL](https://iceberg.apache.org/docs/latest/spark-ddl/)
- [Iceberg Schema Evolution](https://iceberg.apache.org/docs/latest/evolution/)
- [MinIO Quickstart](https://min.io/docs/minio/container/index.html)
- [Armbrust et al. (2021) — Lakehouse paper (CIDR)](https://www.cidrdb.org/cidr2021/papers/cidr2021_paper17.pdf)
- *Designing Data-Intensive Applications*, 2nd Ed. — Ch. 4 & 5